# From training HOGs to OMA GO annotations
Run in order with a Jupyter kernel containing pyOMA 0.15.0 and tqdm. The input is Stage 02’s **unpruned training-HOG OrthoXML**. A training HOG contains at least one CAFA training protein and can also contain OMA and CAFA test proteins. Each distinct protein referenced in a HOG gets one inventory row. Only OMA proteins receive OMA annotation lookups.

The final files are `proteins.tsv`, `oma_full.obo`, and `oma.gaf.gz` in a dated Stage 04 result folder. Use this GAF with the corresponding training XML. If you later run on pruned trees, select that XML and regenerate the cohort or filter the GAF to its retained protein IDs first.

### 1. Select the input and start a run
- `csv` and `gzip` handle tabular text and compressed GAF output. `os.environ` lets the same notebook run with different machine paths.
- `Counter` tallies proteins by source. `closing` closes the pyOMA database on exit, including after an error.
- `Path`, `date` and `count` build readable paths and unique date-based run names. `ParserCreate` streams the XML and `tqdm` displays lookup progress.
- `root` and `analysis` locate the project. `default_xml` identifies the observed local training export. Set `TRAIN_ORTHOXML` to an exact path if your cluster export has a different run name.
- `database` points to the shared May 2026 database. Set `OMA_DATABASE` to override it. The input check stops early if the selected XML is missing.
- `results.mkdir` creates the Stage 04 results parent as needed. The loop tries today’s date first, then adds a numeric suffix if that name exists. It stops as soon as it has created a fresh folder.
- The final print shows which XML and output directory this execution uses.

In [ ]:
import csv, gzip, os
from collections import Counter
from contextlib import closing
from datetime import date
from itertools import count
from pathlib import Path
from xml.parsers.expat import ParserCreate
from tqdm.auto import tqdm

root = Path(os.environ.get("WP1_ROOT", Path.home() / "evomics_projects/saschabrown/workpackage1"))
analysis = root / "HOGprop_validation_project/analysis"
default_xml = analysis / "02_fastoma_output_processing/results/train_orthoxml_a_soga13/cafa_train_unpruned.orthoxml"
xml = Path(os.environ.get("TRAIN_ORTHOXML", default_xml))
database = Path(os.environ.get("OMA_DATABASE", "/home/saschabrown/evomics_projects/Databases/oma-browser/All.May2026/OmaServer.h5"))
if not xml.is_file():
    raise FileNotFoundError(xml)

results = analysis / "04_hogprop/results"
results.mkdir(parents=True, exist_ok=True)
for number in count(1):
    suffix = "" if number == 1 else f"_{number:02d}"
    out = results / f"oma_go_{date.today().isoformat()}{suffix}"
    try:
        out.mkdir()
        break
    except FileExistsError:
        continue
print("Training XML:", xml, "\nOutput:", out)

### 2. List the proteins in the training trees
- `registry` maps each XML gene number to its protein identifier. `members` deduplicates proteins occurring in the trees.
- The parser calls `visit` for each opening tag. Removing the namespace lets it recognize both `gene` and `geneRef`.
- A `gene` declares an identifier. A `geneRef` shows that the gene actually occurs in a HOG. Looking up its number also makes a broken reference fail visibly.
- `ParseFile` reads XML incrementally. The dictionaries and sets are held in memory while the XML tree itself is streamed.
- The suffix identifies the source under this project’s FASTA naming convention. The output retains the exact protein ID so it can be matched back to OrthoXML. Sorting makes the inventory order reproducible.

In [ ]:
def tree_proteins(path):
    registry, members = {}, set()

    def visit(tag, attrs):
        tag = tag.rsplit("}", 1)[-1]
        if tag == "gene":
            registry[int(attrs["id"])] = attrs["protId"]
        elif tag == "geneRef":
            members.add(registry[int(attrs["id"])])
    parser = ParserCreate(namespace_separator="}")
    parser.StartElementHandler = visit
    with path.open("rb") as stream:
        parser.ParseFile(stream)
    return members

members = tree_proteins(xml)
counts = Counter()
with (out / "proteins.tsv").open("w", newline="") as stream:
    writer = csv.writer(stream, delimiter="\t", lineterminator="\n")
    writer.writerow(["protein_id", "source"])
    for protein in sorted(members):
        source = "CAFA_train" if protein.endswith("_train") else "CAFA_test" if protein.endswith("_test") else "OMA"
        writer.writerow([protein, source])
        counts[source] += 1
del members
if not counts["OMA"]:
    raise ValueError("No OMA tree members found; check the selected XML.")
print(dict(counts))

### 3. Copy the OBO bundled with OMA
- `Database` opens the existing HDF5 file using pyOMA. `closing` releases the handle even if a check fails.
- `get_release_name` reads the database release. Check it before exporting so a path change cannot silently mix releases.
- `get_hdf5_handle` gives access to the underlying PyTables file. Reading `Ontologies.GO` retrieves just the embedded OBO bytes.
- The header check catches an unexpected payload. Writing bytes keeps the complete original ontology, including its version, term definitions and relationships.

In [ ]:
from pyoma.browser.db import Database

with closing(Database(str(database))) as db:
    release = db.get_release_name()
    if "may2026" not in "".join(c for c in release.lower() if c.isalnum()):
        raise ValueError(f"Expected OMA May 2026, got {release!r}")
    obo = db.get_hdf5_handle().root.Ontologies.GO.read().tobytes()
if not obo.startswith(b"format-version:") or b"[Term]" not in obo:
    raise ValueError("The embedded GO dataset does not look like OBO text.")
(out / "oma_full.obo").write_bytes(obo)
print(release, next((s.decode() for s in obo.splitlines() if s.startswith(b"data-version:")), "No OBO version header"))

### 4. Look up individual proteins and convert to GAF 2.1
- `oma_ids` streams the saved inventory and yields OMA identifiers only. CAFA labels remain separate for evaluation.
- Open the HDF5 database once. `omaid_to_entry_nr` resolves each OMA protein string to the numeric entry used by pyOMA. An unknown ID raises an error rather than being counted as unannotated.
- `as_gaf=True` asks pyOMA to convert that protein’s annotations into the 17 GAF columns, including protein ID, GO ID, reference, evidence, aspect and taxon. A header alone means pyOMA returned zero eligible annotations. pyOMA filters terms it cannot resolve in its ontology.
- Write the version header once for the combined GAF. Check the header, column count and protein ID before appending each returned row. The counters distinguish proteins with annotations from annotation rows.
- Write gzip output incrementally. The progress bar counts OMA protein lookups, including those with no returned rows. Millions of individual queries can take substantial time. The `.partial` suffix marks an interrupted export; the final rename happens only after the loop succeeds.

In [ ]:
def oma_ids(path):
    with path.open(newline="") as stream:
        for record in csv.DictReader(stream, delimiter="\t"):
            if record["source"] == "OMA":
                yield record["protein_id"]

header = "!gaf-version: 2.1"
partial = out / "oma.gaf.gz.partial"
annotated = rows_written = 0
with closing(Database(str(database))) as db, gzip.open(partial, "wt", encoding="utf-8") as gaf:
    print(header, file=gaf)
    mapper = db.id_mapper["Oma"]
    for protein in tqdm(oma_ids(out / "proteins.tsv"), total=counts["OMA"], unit="protein"):
        entry = mapper.omaid_to_entry_nr(protein)
        version, *rows = db.get_gene_ontology_annotations(entry, as_gaf=True).splitlines()
        if version != header:
            raise ValueError(f"Unexpected GAF header for {protein}: {version!r}")
        for row in rows:
            fields = row.split("\t")
            if len(fields) != 17 or fields[1] != protein:
                raise ValueError(f"Invalid GAF row for {protein}")
            print(row, file=gaf)
        annotated += bool(rows)
        rows_written += len(rows)

partial.replace(out / "oma.gaf.gz")
print(f"{rows_written:,} GAF rows; {annotated:,} annotated OMA proteins; {counts['OMA'] - annotated:,} with no returned rows.")
print(out)

**HOGProp inputs** — use `oma.gaf.gz` for `--gaf`, `oma_full.obo` for `--obo`, and the selected XML for `--oxml`. This exports OMA annotations before HOGProp applies its evidence filter.

**Implementation references** — [pyOMA source](https://github.com/DessimozLab/pyoma) and [the project’s HOGProp GAF reader](../../../software/hogprop/hogprop/Annotations.py).